### Install and import necessary libraries

First, we need to install `selenium` for web scraping and `webdriver_manager` to automatically handle the Chrome WebDriver setup. We also import `pandas` for data handling and `time` for pauses.

In [1]:
%%shell
pip install selenium webdriver_manager pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [2]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print('Libraries imported successfully.')

Libraries imported successfully.


### Set up Chrome WebDriver

Next, we'll configure Selenium to use a headless Chrome browser, which is suitable for environments like Google Colab where there's no graphical interface. We also set up some options to make it run smoothly.

In [5]:
!wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list

!apt-get update
!apt-get install -y google-chrome-stable # Install Chrome browser

# Setup Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless') # Run in headless mode without a UI
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# Add more arguments for stability in headless mode
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920x1080')
chrome_options.add_argument('--incognito')
chrome_options.add_argument('--disable-extensions')
chrome_options.add_argument('--disable-setuid-sandbox')
chrome_options.add_argument('--remote-debugging-port=9222')
chrome_options.add_argument('--log-level=3') # Suppress unnecessary logs
chrome_options.add_argument('--disable-browser-side-navigation')
chrome_options.add_argument('--disable-features=VizDisplayCompositor')

# Specify the path to the Chrome binary explicitly
chrome_options.binary_location = '/usr/bin/google-chrome'

# Initialize Chrome WebDriver
print('Initializing Chrome WebDriver...')
# Use ChromeDriverManager to automatically handle ChromeDriver installation
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
print('WebDriver initialized.')

OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,216 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,041 B in 1s (2,041 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W

### Navigate to the search page and fill in criteria

Now we'll navigate to the specified URL, locate the dropdown menus, and select the required options. We'll also input the start and end dates for the search.

In [6]:
# 1. Go to the search page
url = 'https://www.imprensaoficial.com.br/ENegocios/BuscaENegocios_14_1.aspx'
driver.get(url)
print(f'Navigated to: {url}')

# Wait for the page to load completely
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.ID, 'content_content_content_Negocio_cboArea')))
print('Page loaded and elements are present.')

# 2. Fill in the search criteria in nine dropdown menus
# Select 'Materiais e Equipamentos' for Área
select_area = Select(driver.find_element(By.ID, 'content_content_content_Negocio_cboArea'))
select_area.select_by_visible_text('Materiais e Equipamentos')
print('Selected "Materiais e Equipamentos" for Área.')

# Select 'Generos Alimenticios' for Subárea
# There might be a slight delay for subarea options to load after area selection
time.sleep(2) # Give it some time to update options
select_subarea = Select(driver.find_element(By.ID, 'content_content_content_Negocio_cboSubArea'))
select_subarea.select_by_visible_text('Generos Alimenticios')
print('Selected "Generos Alimenticios" for Subárea.')

# Select 'ENCERRADA' for Status
select_status = Select(driver.find_element(By.ID, 'content_content_content_Status_cboStatus'))
select_status.select_by_visible_text('ENCERRADA')
print('Selected "ENCERRADA" for Status.')

# Set Data Início (01/01/2024)
select_start_day = Select(driver.find_element(By.ID, 'content_content_content_Status_cboAberturaSecaoInicioDia'))
select_start_day.select_by_visible_text('1')

select_start_month = Select(driver.find_element(By.ID, 'content_content_content_Status_cboAberturaSecaoInicioMes'))
select_start_month.select_by_visible_text('1')

select_start_year = Select(driver.find_element(By.ID, 'content_content_content_Status_cboAberturaSecaoInicioAno'))
select_start_year.select_by_visible_text('2024')
print('Set Data Início to 01/01/2024.')

# Set Data Fim (31/12/2024)
select_end_day = Select(driver.find_element(By.ID, 'content_content_content_Status_cboAberturaSecaoFimDia'))
select_end_day.select_by_visible_text('31')

select_end_month = Select(driver.find_element(By.ID, 'content_content_content_Status_cboAberturaSecaoFimMes'))
select_end_month.select_by_visible_text('12')

select_end_year = Select(driver.find_element(By.ID, 'content_content_content_Status_cboAberturaSecaoFimAno'))
select_end_year.select_by_visible_text('2024')
print('Set Data Fim to 31/12/2024.')

# 3. Click the button "Buscar"
search_button = driver.find_element(By.ID, 'content_content_content_btnBuscar')
search_button.click()
print('Clicked the "Buscar" button.')

Navigated to: https://www.imprensaoficial.com.br/ENegocios/BuscaENegocios_14_1.aspx
Page loaded and elements are present.
Selected "Materiais e Equipamentos" for Área.
Selected "Generos Alimenticios" for Subárea.
Selected "ENCERRADA" for Status.
Set Data Início to 01/01/2024.
Set Data Fim to 31/12/2024.
Clicked the "Buscar" button.


### Scrape data from the result table

Now that the search results are displayed, we'll extract the data from the table. We'll handle pagination to scrape data from the first 5 pages, appending all data to a single list.

In [7]:
all_scraped_data = []

# 4. Wait for the result page to load; wait for the complete loading of the result table.
# 5. Scrape the text content in the whole table.
for page_num in range(1, 6): # Loop for the first 5 pages
    print(f'\nScraping page {page_num}...')

    # Wait for the table to be visible and its rows to be present
    WebDriverWait(driver, 20).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="content_content_content_ResultadoBusca_dtgResultadoBusca"]/tbody'))
    )
    WebDriverWait(driver, 20).until(
        EC.visibility_of_element_located((By.XPATH, '//*[@id="content_content_content_ResultadoBusca_dtgResultadoBusca"]/tbody/tr[2]')) # Wait for at least one data row
    )

    table = driver.find_element(By.ID, 'content_content_content_ResultadoBusca_dtgResultadoBusca')
    rows = table.find_elements(By.TAG_NAME, 'tr')

    if page_num == 1: # Get headers only once from the first page
        headers = [header.text for header in rows[0].find_elements(By.TAG_NAME, 'th')]
        print(f'Headers identified: {headers}')
        all_scraped_data.append(headers)

    # Iterate through data rows (skip header row)
    for row in rows[1:]:
        cols = row.find_elements(By.TAG_NAME, 'td')
        if cols: # Ensure it's a data row and not an empty row or footer
            row_data = [col.text for col in cols]
            all_scraped_data.append(row_data)
    print(f'Scraped {len(rows) - 1} rows from page {page_num}.')

    # 6. Go to the next page of the result table and repeat the scraping
    # We will scrape a maximum of 5 pages, or fewer if the 'next' button becomes unavailable.
    if page_num < 5: # Don't try to go to next page after scraping the 5th page
        try:
            # The total number of pages can be found in the span with id 'content_content_content_ResultadoBusca_PaginadorCima_QuantidadedePaginas'
            # Find the 'next page' button by its specific ID
            next_page_button_id = 'content_content_content_ResultadoBusca_PaginadorCima_btnProxima'
            next_page_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.ID, next_page_button_id))
            )

            next_page_button.click()
            print(f'Clicked "próxima >>" button for page {page_num+1}.')

            # Wait for the new page's table to load by checking for its presence again
            WebDriverWait(driver, 10).until(
                EC.staleness_of(table) # Wait for the old table content to disappear
            )
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, 'content_content_content_ResultadoBusca_dtgResultadoBusca')) # Wait for new table element to appear
            )
            time.sleep(2) # Additional short pause to ensure content update and avoid race conditions

        except Exception as e:
            print(f'Could not find the "próxima >>" button or an error occurred on page {page_num}: {e}')
            print('Assuming no more pages or an issue with pagination, stopping.')
            break

print('\nFinished scraping all specified pages.')


Scraping page 1...
Headers identified: []
Scraped 10 rows from page 1.
Clicked "próxima >>" button for page 2.

Scraping page 2...
Scraped 10 rows from page 2.
Clicked "próxima >>" button for page 3.

Scraping page 3...
Scraped 10 rows from page 3.
Clicked "próxima >>" button for page 4.

Scraping page 4...
Scraped 10 rows from page 4.
Clicked "próxima >>" button for page 5.

Scraping page 5...
Scraped 10 rows from page 5.

Finished scraping all specified pages.


### Save data to CSV

Finally, we'll convert the collected list of data into a pandas DataFrame and save it to a CSV file. We'll also make sure to quit the WebDriver instance to free up resources.

In [8]:
# Convert to DataFrame and save to CSV
if all_scraped_data:
    columns_for_df = []
    data_for_df = []

    if all_scraped_data[0]: # If headers were successfully scraped
        columns_for_df = all_scraped_data[0]
        data_for_df = all_scraped_data[1:]
    elif len(all_scraped_data) > 1: # No headers, but there is data
        # Infer number of columns from the first data row
        num_cols = len(all_scraped_data[1])
        columns_for_df = [f'Column_{i+1}' for i in range(num_cols)]
        data_for_df = all_scraped_data[1:]
    else: # Only an empty list in all_scraped_data, or all_scraped_data is just [[]]
        print('No valid data rows found after processing headers.')
        # data_for_df and columns_for_df remain empty, resulting in an empty DataFrame

    if columns_for_df and data_for_df:
        df = pd.DataFrame(data_for_df, columns=columns_for_df)
        csv_filename = 'imprensa_oficial_data.csv'
        df.to_csv(csv_filename, index=False)
        print(f'Data successfully saved to {csv_filename}')
        display(df.head())
    else:
        print('DataFrame is empty or missing columns/data, no data saved.')
else:
    print('No data was scraped.')

# Quit the driver
driver.quit()
print('WebDriver quit.')

Data successfully saved to imprensa_oficial_data.csv


,Column_1,Column_2,Column_3,Column_4,Column_5,Column_6
0,1,"Penitenciária ""Nelson Vieira""",CHAMADA PÚBLICA,003/2024,13/12/2024 10:00,Aquisição de gêneros alimentícios Hortifrutigr...
1,2,Coordenadoria de Infraestrutura e Serviços Esc...,CHAMADA PÚBLICA,04/CP/2024,12/12/2024 15:30,Hortifrútis - interior...
2,3,Coordenadoria de Infraestrutura e Serviços Esc...,CHAMADA PÚBLICA,03/CP/2024,12/12/2024 15:00,Caqui Rama e Goiaba Vermelha...
3,4,Coordenadoria de Infraestrutura e Serviços Esc...,CHAMADA PÚBLICA,02/CP/2024,12/12/2024 14:30,Banana Nanica convencional e orgânico ...
4,5,Coordenadoria de Infraestrutura e Serviços Esc...,CHAMADA PÚBLICA,01/CP/2024,12/12/2024 14:00,Limão Tahiti e Tangerina Ponkan...


WebDriver quit.
